In [32]:
# Load packages
import os

# Set workspace environment variables
os.environ["WORKSPACE_BUCKET"] = "gs://rw-migration-aou-rw-9f879fc4"
os.environ["ARTIFACT_REGISTRY_DOCKER_REPO"] = (
    "us-central1-docker.pkg.dev/all-of-us-rw-prod/aou-rw-gar-remote-repo-docker-prod"
)

# Load the remaining workspace environment settings
%run Setting_Env_Variables.ipynb

Successfully set WORKSPACE_CDR to: wb-silky-artichoke-2408.C2024Q3R9
Found bucket: id=dataproc-staging-aou-rw-9f879fc4, bucketName=dataproc-staging-wb-fast-radish-6415
Found bucket: id=dataproc-temp-aou-rw-9f879fc4, bucketName=dataproc-temp-wb-fast-radish-6415
Found bucket: id=rw-migration-aou-rw-9f879fc4, bucketName=rw-migration-aou-rw-9f879fc4
Found bucket: id=vwb-aou-datasets-controlled-v8r9, bucketName=vwb-aou-datasets-controlled

Variables extracted:
GOOGLE_CLOUD_PROJECT: wb-fast-radish-6415
WORKSPACE_BUCKET: gs://rw-migration-aou-rw-9f879fc4
WORKSPACE_TEMP_BUCKET: NOT FOUND
WORKSPACE_CDR: wb-silky-artichoke-2408.C2024Q3R9

✅ Saved to /home/dataproc/.bashrc
C2024Q3R9 BQ_DATASET
JupyterLab_Spark_cluster_for_AoU_Spark_20261021 DATAPROC_CLUSTER
dataproc-staging-aou-rw-9f879fc4 GCS_BUCKET
dataproc-temp-aou-rw-9f879fc4 GCS_BUCKET
rw-migration-aou-rw-9f879fc4 GCS_BUCKET
vwb-aou-datasets-controlled-v8r9 GCS_BUCKET
GOOGLE_CLOUD_PROJECT = wb-fast-radish-6415
WORKSPACE_BUCKET = gs://rw-migr

In [23]:
%%writefile missing.sh

plink2 \
    --bfile "${GENO}/EUR_passed_chr${CHROM}" \
    --missing \
    --memory 3100 \
    --out "${OUTPUT}/EUR_passed_chr${CHROM}"


Overwriting missing.sh


In [24]:
# Logging directory
LOGGING=f"{os.environ['WORKSPACE_BUCKET']}/data/logging"
# Output directory 
OUTPUT=f"{os.environ['WORKSPACE_BUCKET']}/data/chip/plink"
# Path to genomic data 
MNT_PATH="/mnt/data/input/gs/rw-migration-aou-rw-9f879fc4/data/chip/plink"

DISK_SIZE = 4 
    
# Loop to submit dsub command 
for CHROM in [2, 4, 15]:    
    
    print(f"Submitting dsub command for chromosome {CHROM}")

    BED=f'gs://rw-migration-aou-rw-9f879fc4/data/chip/plink/EUR_passed_chr{CHROM}.bed'
    BIM=f'gs://rw-migration-aou-rw-9f879fc4/data/chip/plink/EUR_passed_chr{CHROM}.bim'
    FAM=f'gs://rw-migration-aou-rw-9f879fc4/data/chip/plink/EUR_passed_chr{CHROM}.fam'
    
    # DSUB command 
    command = [
        "aou_dsub",
        "--project", os.getenv('GOOGLE_CLOUD_PROJECT'), 
        "--user-project", os.getenv('GOOGLE_CLOUD_PROJECT'), 
        "--image", f"{os.environ['ARTIFACT_REGISTRY_DOCKER_REPO']}/gfanz/plink2:latest",
        "--disk-size", str(DISK_SIZE),
        "--machine-type", "n1-standard-1",
        "--logging", LOGGING,
        "--input", f"BED={BED}",
        "--input", f"BIM={BIM}",
        "--input", f"FAM={FAM}",
        "--output-recursive", f"OUTPUT={OUTPUT}",
        "--env", f"GENO={MNT_PATH}",
        "--env", f"CHROM={CHROM}",
        "--script", "missing.sh"
    ]

    # Run in same shell as source command
    dsub_command = subprocess.run(
        f"source ~/aou_dsub.bash && {' '.join(command)}",
        shell=True,
        executable="/bin/bash",
        capture_output=True,
        text=True
    )

    print(dsub_command.stdout)
    print(dsub_command.stderr)

Submitting dsub command for chromosome 2
missing--vienose--261022-211301-96

Job properties:
  job-id: missing--vienose--261022-211301-96
  job-name: missing
  user-id: vienose
Provider internal-id (operation): projects/wb-fast-radish-6415/locations/us-central1/jobs/missing--vienose--261022-211301-96-0-0
Launched job-id: missing--vienose--261022-211301-96
To check the status, run:
  dstat --provider google-batch --project wb-fast-radish-6415 --location us-central1 --jobs 'missing--vienose--261022-211301-96' --users 'vienose' --status '*'
To cancel the job, run:
  ddel --provider google-batch --project wb-fast-radish-6415 --location us-central1 --jobs 'missing--vienose--261022-211301-96' --users 'vienose'

Submitting dsub command for chromosome 4
missing--vienose--261022-211303-96

Job properties:
  job-id: missing--vienose--261022-211303-96
  job-name: missing
  user-id: vienose
Provider internal-id (operation): projects/wb-fast-radish-6415/locations/us-central1/jobs/missing--vienose--

In [ ]:

import os
import subprocess
import pandas as pd 
from functools import reduce
from io import StringIO
import shutil
from pathlib import Path
import gc 

for CHROM in [2, 4, 15, 17]:

    PATH = f"{os.environ['WORKSPACE_BUCKET']}/data/chip/plink/EUR_passed_chr{CHROM}.vmiss"
    result = subprocess.run(
        ["gcloud", "storage", "cat", PATH,
         f"--billing-project={os.environ['GOOGLE_CLOUD_PROJECT']}"],
        capture_output=True, text=True, check=True
    )
    df1 = pd.read_csv(StringIO(result.stdout), sep="\t")[['ID', 'MISSING_CT']]

    PATH = f"{os.environ['WORKSPACE_BUCKET']}/data/chip/plink/EUR_ab0.35_dp20_call0.9_hwe1e15_chr{CHROM}.vmiss"
    result = subprocess.run(
        ["gcloud", "storage", "cat", PATH,
         f"--billing-project={os.environ['GOOGLE_CLOUD_PROJECT']}"],
        capture_output=True, text=True, check=True
    )
    df2 = pd.read_csv(StringIO(result.stdout), sep="\t")[['ID', 'MISSING_CT']]
    
    df2.rename(columns={'MISSING_CT':'MISSING_CT_FILTER'}, inplace=True)
    
    df2['ID'] = 'chr' + df2['ID']
    
    df = pd.merge(df1, df2, on=['ID'])
    
    check = len(df[df['MISSING_CT'] > df['MISSING_CT_FILTER']]) 
    
    if check > 0: 
        print("ERROR") 
        
    check = len(df[df['MISSING_CT'] < df['MISSING_CT_FILTER']]) 
    
    if not check > 0: 
        print("ERROR") 
        
    print(F"Chromosome {CHROM} was filtered correctly ....") 
        
print("All chromosomes were filtered correctly ....") 
    